In [1]:
from documents_large import DOCUMENTS_LARGE, DOCUMENT_NAMES
import numpy as np

def print_section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def split_documents_into_chunks(documents, document_names):
    chunks = []
    chunk_metadata = []

    for document_id, (document, document_name) in enumerate(zip(documents, document_names)):
        lines = document.strip().split("\n")

        for line_id, line in enumerate(lines):
            line = line.strip()

            if not line:
                continue

            chunks.append(line)

            chunk_metadata.append({
                "chunk_id": len(chunks) - 1,
                "source_document_id": document_id,
                "source_document_name": document_name,
                "line_id": line_id,
            })

    return chunks, chunk_metadata


print_section("STEP 1 — LOADING DOCUMENTS")

print("Number of loaded documents:", len(DOCUMENTS_LARGE))
print()

for i, name in enumerate(DOCUMENT_NAMES):
    print(f"Document {i}: {name}")


print_section("STEP 2 — CREATING CHUNKS")

chunks, chunk_metadata = split_documents_into_chunks(
    documents=DOCUMENTS_LARGE,
    document_names=DOCUMENT_NAMES
)

print("Number of created chunks:", len(chunks))
print()

print("Chunks per document:")
for document_name in DOCUMENT_NAMES:
    count = sum(
        1 for metadata in chunk_metadata
        if metadata["source_document_name"] == document_name
    )
    print(f"- {document_name}: {count} chunks")


print_section("FIRST 10 CHUNKS")

for i in range(min(10, len(chunks))):
    print("-" * 80)
    print("Chunk ID:", chunk_metadata[i]["chunk_id"])
    print("Source document:", chunk_metadata[i]["source_document_name"])
    print("Line ID:", chunk_metadata[i]["line_id"])
    print("Text:", chunks[i])


STEP 1 — LOADING DOCUMENTS
Number of loaded documents: 6

Document 0: employee_records
Document 1: coffee_inventory
Document 2: apple_pie_recipe
Document 3: phone_book
Document 4: retrieve_top_k_documentation
Document 5: general_facts

STEP 2 — CREATING CHUNKS
Number of created chunks: 181

Chunks per document:
- employee_records: 31 chunks
- coffee_inventory: 30 chunks
- apple_pie_recipe: 30 chunks
- phone_book: 30 chunks
- retrieve_top_k_documentation: 30 chunks
- general_facts: 30 chunks

FIRST 10 CHUNKS
--------------------------------------------------------------------------------
Chunk ID: 0
Source document: employee_records
Line ID: 0
Text: 1. Alice Morgan is 29 years old, works as a Backend Developer at Northstar Analytics, and earns a gross monthly salary of 7,800 USD.
--------------------------------------------------------------------------------
Chunk ID: 1
Source document: employee_records
Line ID: 1
Text: 2. Benjamin Carter is 41 years old, works as a Senior Data Engine

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-mpnet-base-v2"
)

word1 = "biały kot"
word2 = "kot jest biały"

emb1 = model.encode(
    word1,
    normalize_embeddings=True
)

emb2 = model.encode(
    word2,
    normalize_embeddings=True
)

print("biały kot")
print(emb1[:15])

print()
print("kot jest biały")
print(emb2[:15])

similarity = np.dot(
    emb1,
    emb2
)

print("Similarity:", similarity)


print_section("STEP 3 - CREATE CHUNK EMBEDDINGS")

chunk_embeddings = model.encode(
    chunks,
    normalize_embeddings=True
)

print("Embedding matrix shape:")
print(chunk_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


biały kot
[ 0.00517607 -0.01932392  0.02338176 -0.01319121  0.00567496  0.04107568
  0.0035731   0.04845952  0.04824371  0.04205294  0.01719831  0.0373252
  0.01733477  0.03612221 -0.00660144]

kot jest biały
[ 0.00443334  0.01816     0.03314075 -0.01174799  0.00675404  0.02025125
 -0.0064068   0.04595991  0.0725126   0.01943766  0.02064018  0.02672752
  0.01943173 -0.00038112  0.01242456]
Similarity: 0.896149

STEP 3 - CREATE CHUNK EMBEDDINGS
Embedding matrix shape:
(181, 768)


In [3]:
print_section("FIRST EMBEDDING")

print("Chunk:")
print(chunks[0])

print()
print("Embedding length:")
print(len(chunk_embeddings[0]))

print()
print("First 20 dimensions:")
print(chunk_embeddings[0][:20])


FIRST EMBEDDING
Chunk:
1. Alice Morgan is 29 years old, works as a Backend Developer at Northstar Analytics, and earns a gross monthly salary of 7,800 USD.

Embedding length:
768

First 20 dimensions:
[-0.00219296  0.0704857  -0.00450463  0.01580825 -0.00401145  0.0208445
  0.03426288 -0.00871004 -0.02008277 -0.0206442   0.02796097  0.04682217
  0.02359392  0.1106318  -0.03166721  0.04786775 -0.02129411  0.01853939
  0.08456504  0.01213513]


In [4]:
query = """
Who works as a Security Analyst at Northstar Analytics?
"""

print_section("STEP 4 - QUERY EMBEDDING")

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

print("Query:")
print(query)

print()
print("Embedding length:")
print(len(query_embedding))

print()
print("First 20 dimensions:")
print(query_embedding[:20])


STEP 4 - QUERY EMBEDDING
Query:

Who works as a Security Analyst at Northstar Analytics?


Embedding length:
768

First 20 dimensions:
[ 0.00932254  0.06091205 -0.03843472  0.01517162 -0.03521221  0.00023317
  0.09937507 -0.08985045 -0.03127094 -0.01057563  0.0164823  -0.00940769
  0.00344195  0.08664489  0.03151252  0.09071548  0.01986738 -0.00430035
 -0.01659558  0.02416872]


In [5]:
similarity = np.dot(
    chunk_embedding,
    query_embedding
)

print("Similarity:", similarity)

NameError: name 'chunk_embedding' is not defined